In [4]:
import os
import shutil
import bioread
import neurokit2 as nk
import pandas as pd

# --- PARAMETERS TO CHANGE ---
# CLICK on folder icon on the side panel to open file view
# CREATE a new folder called DATA using the folder with plus sign icon
# GO in the DATA folder and CLICK on the upward pointing arrow to upload the ACQ data files to analyze
# For large files, you will need to manually confirm the upload for each one of them
# Once the files are uploaded, wait a few minutes before running the code to avoid errors (large files need time to fully upload)
input_folder = "DATA" # Folder containing the ACQ files to analyze
processed_folder = "DATA_processed" # Folder that will contain all the processed data files (automatically created by the script)
file_name_keyword = "Habituation" # CHANGE the keyword to a word present in all the files to analyze (e.g. "Habituation" for files like this: 068A_03P_Habituation_Rouge_J1.acq)
output_csv = "HRV_trauma5min.csv" # NAME the excel that will contains all HRV data (1 row = 1 participant)
file_length = 250 # CHANGE length requirement for the files to be inlcuded (in seconds) 
end_time = 250 # CHANGE length of recording time to be analyzed (in seconds)
start_time = 0 # ADD start time if you need to analyze only a segment, otherwise leave at 0 (in seconds)
# ----------------------------

# Compute duration
duration = end_time-start_time

# Créer le dossier de sortie si nécessaire
os.makedirs(processed_folder, exist_ok=True)

# liste des fichiers .acq
files = [f for f in os.listdir(input_folder) if file_name_keyword in f]

# Si un fichier CSV existe déjà, on le recharge pour éviter d'écraser
if os.path.exists(output_csv):
    df_results = pd.read_csv(output_csv)
else:
    df_results = pd.DataFrame()

print("READY, SET, GO!")

for f in files:
    filepath = os.path.join(input_folder, f)
    try:
        data = bioread.read_file(filepath)
    except Exception as e:
        print(f"Erreur lecture {f} : {e}")
        continue

    # Chercher le canal ECG
    ecg_channel = None
    for ch in data.channels:
        if "ECG" in ch.name.upper():
            ecg_channel = ch
            break
    if ecg_channel is None:
        print(f"⚠️ Aucun canal ECG trouvé dans {f}")
        continue

    ecg_signal = ecg_channel.data
    ecg_time = ecg_channel.time_index
    sampling_rate = ecg_channel.samples_per_second

    if ecg_time[-1] >= file_length:
    # Conditions to select only file with enough recording time (e.g., at least 250 sec)
    #If condition then tab

        try:
            # Analyse ECG avec NeuroKit2
            signals, info = nk.ecg_process(ecg_signal, sampling_rate=sampling_rate, method="neurokit")
    
            # HRV (time + frequency + nonlinear) sur les 2 premières minutes
            if not start_time:
                mask = ecg_time <= end_time
            else:
                mask = (ecg_time <= end_time) & (ecg_time >= start_time) # If you need to only analyse a segment
            
            hrv_metrics = nk.hrv(signals.ECG_R_Peaks[mask],
                                 sampling_rate=sampling_rate,
                                 show=False)
    
            # Ajouter ID + durée totale
            hrv_metrics["Participant"] = os.path.splitext(f)[0]
            hrv_metrics["RecordingLength"] = ecg_time[-1]
            hrv_metrics["AnalysisLength"] = duration # add the duration time in the output dataframe
    
            # Ajouter dans le DataFrame global
            df_results = pd.concat([df_results, hrv_metrics], ignore_index=True)
    
            # Sauvegarder le CSV après chaque fichier
            df_results.to_csv(output_csv, index=False)
            print(f"✅ Résultats sauvegardés pour {f}")
    
            # Déplacer le fichier dans processed_folder
            shutil.move(filepath, os.path.join(processed_folder, f))
            print(f"📂 {f} déplacé vers {processed_folder}")
    
        except Exception as e:
            print(f"Erreur traitement {f} : {e}")
            continue
            
print("DONE!")

READY, SET, GO!
✅ Résultats sauvegardés pour 077A_03P_Habituation_Rouge_J1.acq
📂 077A_03P_Habituation_Rouge_J1.acq déplacé vers DATA_processed
✅ Résultats sauvegardés pour 329A_02P_Habituation_Bleu_J1.acq
📂 329A_02P_Habituation_Bleu_J1.acq déplacé vers DATA_processed
✅ Résultats sauvegardés pour 073A_07P_Habituation_Rouge_J1.acq
📂 073A_07P_Habituation_Rouge_J1.acq déplacé vers DATA_processed
✅ Résultats sauvegardés pour 074A_02P_Habituation_Bleu_J1.acq
📂 074A_02P_Habituation_Bleu_J1.acq déplacé vers DATA_processed
✅ Résultats sauvegardés pour 288A_03P_Habituation_Rouge_J1.acq
📂 288A_03P_Habituation_Rouge_J1.acq déplacé vers DATA_processed
✅ Résultats sauvegardés pour 070A_02P_Habituation_Bleu_J1.acq
📂 070A_02P_Habituation_Bleu_J1.acq déplacé vers DATA_processed
✅ Résultats sauvegardés pour 068A_03P_Habituation_Rouge_J1.acq
📂 068A_03P_Habituation_Rouge_J1.acq déplacé vers DATA_processed
DONE!
